## Load segments from Slicer

In [ ]:
import numpy
import sys
import os
import matplotlib.pyplot as plt
# import skimage
import nrrd
from scipy import ndimage
import meshio

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import getSurfaceMesh, tetra_mesh_from_stl, tetra_shell_from_two_surfaces, set_the_offset


pixel_spacing = numpy.loadtxt("/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/pixel_spacing.txt")

base = "/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/"

# offset = numpy.loadtxt("/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/offset.txt")


in_folder = "segment_final/"    

folder = "Meshes/"

if not os.path.isdir(base + folder):
    os.mkdir(base + folder)




filename = "Segmentation_1.seg.nrrd"
export_name = base + folder + filename[:-8]

eyeball_id = 1
background_id = -1


data, header = nrrd.read(base + in_folder + filename)
print(data.shape)



id_max = numpy.max(data)

print("pixel spacing = ",pixel_spacing)
print("id = ", id_max )

print("dimensions = ", data.shape)

if pixel_spacing[2]!= pixel_spacing[1]:
    print("resampling")
    factor = pixel_spacing[2]/pixel_spacing[1]
    data = ndimage.zoom(data, [1,1,factor], order = 0)

data = ndimage.median_filter(data, size=1)



print("dimensions = ", data.shape)

id = int(numpy.max(data))
print("id = ", id)




FileNotFoundError: [Errno 2] No such file or directory: '/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/segment_final/Segmentation_2.seg.nrrd'

In [ ]:
for id in range(1,numpy.max(data)+1):
    if id != background_id:
    # for id in range(3,4):
        print("id = ", id)


        if len(numpy.where((data>id-0.5) & (data<id+0.5 ))[0])>0:

            mask = numpy.zeros(data.shape, dtype=numpy.uint8)
            mask[(data>id-0.5) & (data<id+0.5 )]=1

            print(len(numpy.where(mask==1)[0]))

            mask = ndimage.median_filter(mask, size=5)
            mask[mask<0.5]=0
            mask[mask>=0.5]=1

            numpy.save(export_name +"_filtered_surf_id_"+str(id), mask)

            if id ==eyeball_id:
                mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing[0], True )

                mask_voxels = numpy.argwhere(mask == 1)  # shape (N, 3), coordinates in (z, y, x)
                # compute mean voxel position
                offset = mask_voxels.mean(axis=0)*pixel_spacing[0]  # [z_mean, y_mean, x_mean]
                numpy.savetxt(base + folder + "offset.txt", offset)

            else:
                mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing[0], False )



id =  1
258250
Problem edges: 0
Outer and inner surfaces written.
id =  2
22539
Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/601_PDT1_0.5_Fat/Meshes/Segmentation_2._surf_id_2.stl
id =  3
26474
Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/601_PDT1_0.5_Fat/Meshes/Segmentation_2._surf_id_3.stl
id =  4
21717
Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/601_PDT1_0.5_Fat/Meshes/Segmentation_2._surf_id_4.stl
id =  5
26254
Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/601_PDT1_0.5_Fat/Meshes/Segmentation_2._surf_id_5.stl
id =  6
27350
Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/601_PDT1_0.5_Fat/Meshes/Segmentation_2._surf_id_6.stl
id =  7
2198
Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/601_PDT1_0.5_Fat/Meshes/Segme

In [ ]:
h=1.0

out_folder = "re-meshed_h_"+str(h)+"/"

if not os.path.isdir(base + folder + out_folder):
    os.mkdir(base + folder + out_folder)


for id in range(1,numpy.max(data)+1):
    

    if (id !=eyeball_id) and (id != background_id):
        print("id = ", id) 
        

        stl_file = export_name +"_surf_id_"+str(id)+".stl"

        if os.path.isfile(stl_file):
            tetra_mesh_from_stl(stl_file, base + folder + out_folder + "id_"+str(id)+"_" + "_h_"+str(h), h)
            set_the_offset(base + folder + out_folder + "id_"+str(id)+"_" + "_h_"+str(h), offset)


id =  2
Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/601_PDT1_0.5_Fat/Meshes/Segmentation_2._surf_id_2.stl'...
Info    : 18480 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/601_PDT1_0.5_Fat/Meshes/Segmentation_2._surf_id_2.stl'
Info    : Classifying surfaces (angle: 70)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 18480 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    : Found 2 model surfaces
Info    : Found 1 model curves
Info    : Done classifying surfaces (Wall 0.177119s, CPU 0.171862s)
Info    : Creating geometry of discrete curves...
Info    : Done creating geometry of discrete curves (Wall 7.08178e-06s, CPU 7e-06s)
Info    : Creating geometry of discrete surfaces...
Info    : Done creating geometry of discrete surfaces (Wall 0.0712066s, CPU 0.068344s)               

Warning: STL can only write triangle cells. Discarding tetra, vertex, line.

id =  3
Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/601_PDT1_0.5_Fat/Meshes/Segmentation_2._surf_id_3.stl'...
Info    : 17756 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/601_PDT1_0.5_Fat/Meshes/Segmentation_2._surf_id_3.stl'
Info    : Classifying surfaces (angle: 70)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 17756 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    :  - Level 1 partition with 8850 triangles split in 2 parts because parametrized triangles are too small (9.61166e-09)
Info    : Found 3 model surfaces
Info    : Found 2 model curves
Info    : Done classifying surfaces (Wall 0.209759s, CPU 0.204026s)
Info    : Creating geometry of discrete curves...
Info    : Done creating geometry of discrete curves (Wall 1.32918e-05s, CPU 1e-05s)
Info    : Creating geometr

Warning: STL can only write triangle cells. Discarding tetra, vertex, line.

id =  4
Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/601_PDT1_0.5_Fat/Meshes/Segmentation_2._surf_id_4.stl'...
Info    : 15960 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/601_PDT1_0.5_Fat/Meshes/Segmentation_2._surf_id_4.stl'
Info    : Classifying surfaces (angle: 70)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 15960 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    :  - Level 1 partition with 7975 triangles split in 2 parts because parametrized triangles are too small (8.35782e-09)
Info    : Found 3 model surfaces
Info    : Found 2 model curves
Info    : Done classifying surfaces (Wall 0.191869s, CPU 0.186871s)
Info    : Creating geometry of discrete curves...
Info    : Done creating geometry of discrete curves (Wall 1.21649e-05s, CPU 1e-05s)
Info    : Creating geometr

...

Warning: STL can only write triangle cells. Discarding tetra, vertex, line.

id =  6
Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/601_PDT1_0.5_Fat/Meshes/Segmentation_2._surf_id_6.stl'...
Info    : 18208 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/601_PDT1_0.5_Fat/Meshes/Segmentation_2._surf_id_6.stl'
Info    : Classifying surfaces (angle: 70)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 18208 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    :  - Level 1 partition with 9004 triangles split in 2 parts because parametrized triangles are too small (8.26001e-09)
Info    : Found 3 model surfaces
Info    : Found 2 model curves
Info    : Done classifying surfaces (Wall 0.222012s, CPU 0.214171s)
Info    : Creating geometry of discrete curves...
Info    : Done creating geometry of discrete curves (Wall 8.37445e-06s, CPU 8e-06s)
Info    : Creating geometr

Warning: STL can only write triangle cells. Discarding tetra, vertex, line.

id =  7
Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/601_PDT1_0.5_Fat/Meshes/Segmentation_2._surf_id_7.stl'...
Info    : 2444 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/601_PDT1_0.5_Fat/Meshes/Segmentation_2._surf_id_7.stl'
Info    : Classifying surfaces (angle: 70)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 2444 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    : Found 2 model surfaces
Info    : Found 1 model curves
Info    : Done classifying surfaces (Wall 0.0236148s, CPU 0.022214s)
Info    : Creating geometry of discrete curves...
Info    : Done creating geometry of discrete curves (Wall 4.95464e-06s, CPU 4e-06s)
Info    : Creating geometry of discrete surfaces...
Info    : Done creating geometry of discrete surfaces (Wall 0.0102291s, CPU 0.009193s)                

Warning: STL can only write triangle cells. Discarding tetra, vertex, line.

In [ ]:
stl_file = export_name +"_surf_id_"+str(eyeball_id)+".stl"


tetra_shell_from_two_surfaces(stl_file.replace(".stl", "_outer.stl"), stl_file.replace(".stl", "_inner.stl"), base + folder + out_folder + "id_"+str(eyeball_id)+"_shell_" + "_h_"+str(h), h)
set_the_offset(base + folder + out_folder + "id_"+str(eyeball_id)+"_shell_" + "_h_"+str(h), offset)


Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/601_PDT1_0.5_Fat/Meshes/Segmentation_2._surf_id_1_outer.stl'...
Info    : 59436 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/601_PDT1_0.5_Fat/Meshes/Segmentation_2._surf_id_1_outer.stl'
Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/601_PDT1_0.5_Fat/Meshes/Segmentation_2._surf_id_1_inner.stl'...
Info    : 59436 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/601_PDT1_0.5_Fat/Meshes/Segmentation_2._surf_id_1_inner.stl'
Info    : Classifying surfaces (angle: 40)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 59436 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    :  - Level 0 partition with 59436 triangles 

Warning: STL can only write triangle cells. Discarding tetra, vertex, line.